# Amir's modelling experiment: 43 selected features

This notebook follows `templates/modelling.ipynb` and uses the shared
`BattleExperiment` runner. It keeps Amir's 43 numeric features and compares
Logistic Regression with Histogram Gradient Boosting.

**Workflow:** load training features → compare both candidates using five-fold
cross-validation (CV) on development data → explicitly evaluate the chosen model
on the holdout → optionally refit on all labelled battles and create a submission.

Holdout evaluation and submission generation are **off by default**. All artifacts
are saved under a unique `outputs/amirhosein/<run>/` folder in this checkout.
Previous scores from the standalone notebook do not apply to this workflow.


In [ ]:
from pathlib import Path
from datetime import datetime
from collections import Counter
import sys

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "pyproject.toml").is_file()
             and (p / "src/pokemon_challenge").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open this repository folder or run from a notebook inside it.")
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from pokemon_challenge.experiment import BattleExperiment, Candidate
from pokemon_challenge.paths import data_dir

OWNER = "amirhosein"
RUN = datetime.now().strftime("%Y%m%d_%H%M%S_%f_selected_features")
DATA_DIR = data_dir()  # data/raw/, data/, Kaggle, or POKEMON_DATA_DIR
print("Project:", ROOT)
print("Data:", DATA_DIR)
print("Run:", OWNER, RUN)


## Context & Methods

There are **43 numeric inputs**: the 26 explicitly named features plus one count
for each of the 17 real elemental types. `notype` is a missing-secondary-type
placeholder and is excluded. The requested `p1-switch` spelling is normalized to
`p1_switch`. No species identities, outcome-derived rates or extra features enter
the models.

| Features | Definition |
| --- | --- |
| `p1_mean_{hp,atk,def,spa,spd,spe}` | Mean base stat over P1's roster |
| `p1_type_{type}_count` | Number of P1 roster members carrying each real type |
| `p1_team_diversity` | Number of distinct real types in P1's roster |
| `p2_lead_{hp,atk,def,spa,spd,spe}` | Supplied P2 lead's base stats |
| `p1_alive_at_turn30`, `p2_alive_at_turn30` | Six minus distinct Pokémon observed fainted |
| `alive_diff_at_turn30` | P1 remaining count minus P2 remaining count |
| `p1_mean_boosts`, `p2_mean_boosts` | Mean across turns of the sum of the five boost stages |
| `p1_no_of_moves`, `p2_no_of_moves` | Number of non-null recorded moves |
| `p1_dmg`, `p2_dmg` | Positive observed opponent HP drops, summed by Pokémon |
| `p1_heal`, `p2_heal` | Positive observed own HP increases, summed by Pokémon |
| `p1_switch`, `p2_switch` | Consecutive active-name changes, including forced replacements |

### Key assumptions

All timeline features stop after the first 30 entries. Fainted means zero HP or
status `fnt`, with each name counted once. Sleep and freeze do not imply fainting;
unseen members are presumed remaining in a six-member team.

Damage/healing compare the same Pokémon's successive observations, including
returns after switching. A newly seen Pokémon starts from an assumed HP of 1.0,
matching the earlier project's depletion proxy. These totals include changes
from unobserved intervals and do not prove which move caused them. HP units are
fractions of a Pokémon's maximum, not absolute HP points.

### Shared feature interface

`extract_selected_features` takes one battle and returns the same 43 numeric keys
for every observation. The shared runner removes the ID and target before calling
it and prefixes its output columns with `selected__`. No learned statistics or
outcome-derived rates are computed here.


In [ ]:
TEAM_SIZE = 6
MAX_OBSERVED_TURNS = 30

STATS = ('hp', 'atk', 'def', 'spa', 'spd', 'spe')
TYPES = (
    'bug', 'dark', 'dragon', 'electric', 'fighting', 'fire', 'flying',
    'ghost', 'grass', 'ground', 'ice', 'normal', 'poison', 'psychic',
    'rock', 'steel', 'water',
)
BOOST_STATS = ('atk', 'def', 'spa', 'spd', 'spe')
FEATURE_COLUMNS = (
    [f'p1_mean_{stat}' for stat in STATS]
    + [f'p1_type_{kind}_count' for kind in TYPES]
    + ['p1_team_diversity']
    + [f'p2_lead_{stat}' for stat in STATS]
    + ['p1_alive_at_turn30', 'p2_alive_at_turn30', 'alive_diff_at_turn30']
    + ['p1_mean_boosts', 'p2_mean_boosts']
    + ['p1_no_of_moves', 'p2_no_of_moves']
    + ['p1_dmg', 'p1_heal', 'p2_dmg', 'p2_heal', 'p1_switch', 'p2_switch']
)
assert len(FEATURE_COLUMNS) == len(set(FEATURE_COLUMNS)) == 43


def extract_selected_features(battle):
    roster = battle.get('p1_team_details') or []
    lead = battle.get('p2_lead_details') or {}
    timeline = (battle.get('battle_timeline') or [])[:MAX_OBSERVED_TURNS]
    features = {}
    for stat in STATS:
        features[f'p1_mean_{stat}'] = float(np.mean([
            pokemon.get(f'base_{stat}', 0) for pokemon in roster
        ])) if roster else 0.0
    type_counts = Counter(
        str(kind).lower()
        for pokemon in roster
        for kind in pokemon.get('types', [])
        if str(kind).lower() != 'notype'
    )
    unknown_types = set(type_counts) - set(TYPES)
    if unknown_types:
        raise ValueError(f'Unknown real roster types: {sorted(unknown_types)}')
    for kind in TYPES:
        features[f'p1_type_{kind}_count'] = type_counts[kind]
    features['p1_team_diversity'] = len(type_counts)
    for stat in STATS:
        features[f'p2_lead_{stat}'] = lead.get(f'base_{stat}', 0)

    fainted = {'p1': set(), 'p2': set()}
    previous_hp = {'p1': {}, 'p2': {}}
    previous_name = {'p1': None, 'p2': None}
    boost_sums = {'p1': [], 'p2': []}
    for side in ('p1', 'p2'):
        features[f'{side}_no_of_moves'] = 0
        features[f'{side}_dmg'] = 0.0
        features[f'{side}_heal'] = 0.0
        features[f'{side}_switch'] = 0

    for turn in timeline:
        for side in ('p1', 'p2'):
            opponent = 'p2' if side == 'p1' else 'p1'
            state = turn.get(f'{side}_pokemon_state') or {}
            name = state.get('name')
            boosts = state.get('boosts') or {}
            boost_sums[side].append(sum(boosts.get(stat, 0) for stat in BOOST_STATS))
            features[f'{side}_no_of_moves'] += turn.get(f'{side}_move_details') is not None
            if not name:
                previous_name[side] = None
                continue
            if previous_name[side] is not None and previous_name[side] != name:
                features[f'{side}_switch'] += 1
            previous_name[side] = name
            hp = state.get('hp_pct')
            if hp == 0 or state.get('status') == 'fnt':
                fainted[side].add(name)
            if hp is not None:
                hp = float(hp)
                old_hp = previous_hp[side].get(name, 1.0)
                features[f'{opponent}_dmg'] += max(old_hp - hp, 0.0)
                features[f'{side}_heal'] += max(hp - old_hp, 0.0)
                previous_hp[side][name] = hp

    for side in ('p1', 'p2'):
        features[f'{side}_alive_at_turn30'] = TEAM_SIZE - len(fainted[side])
        features[f'{side}_mean_boosts'] = float(np.mean(boost_sums[side])) if boost_sums[side] else 0.0
    features['alive_diff_at_turn30'] = features['p1_alive_at_turn30'] - features['p2_alive_at_turn30']
    assert set(features) == set(FEATURE_COLUMNS)
    return features



FEATURE_FUNCTIONS = {"selected": extract_selected_features}
print("Selected numeric inputs:", len(FEATURE_COLUMNS))
display(pd.DataFrame({"feature": FEATURE_COLUMNS}))


## 2. Load features and use the shared validation settings

Inputs are located by the project's shared helper: local `data/raw/` or `data/`,
the supported Kaggle paths, or an explicit `POKEMON_DATA_DIR`. Dataset location
never determines the output directory. To reuse a dataset stored elsewhere, set
`POKEMON_DATA_DIR` to that folder before starting the notebook kernel.

The unchanged `configs/validation.json` defaults reserve 20% of labelled battles
as a holdout and use five stratified folds within the other 80%, with seed 42.
Every candidate uses the same saved assignments. Imputation, constant removal,
and Logistic Regression scaling are fitted using each fold's training rows only.

**Validation change from the old notebook:** this version uses all training rows
and the existing shared splitter. It does not exclude training/test content matches
or group duplicate inputs. Duplicate-policy improvements are deferred; the shared
validation code is unchanged. Old grouped-CV and holdout scores are not comparable
to results from this setup. Test data is read only when generating a submission.

Rerun setup to obtain a new run name before constructing a new experiment.


In [ ]:
experiment = BattleExperiment(
    owner=OWNER,
    run=RUN,
    feature_functions=FEATURE_FUNCTIONS,
    data_path=DATA_DIR,
)
assert experiment.X.shape[1] == 43
assert set(experiment.X.columns) == {f"selected__{name}" for name in FEATURE_COLUMNS}
if not np.isfinite(experiment.X.to_numpy()).all():
    raise ValueError("All selected training features must be finite numeric values.")

display(experiment.X.head())
print("Numeric input features:", experiment.X.shape[1])
display(experiment.assignments.groupby(["partition", "validation_fold"])
        .size().rename("battles").to_frame())


## 3. Compare the original two model configurations

Both candidates use the same `selected` feature group and retain the standalone
notebook's classifier parameters. The runner adds its standard fold-fitted
preprocessing. `percentile=100` keeps all nonconstant inputs; automatic feature
selection is disabled.

**CV (cross-validation):** train on four development folds and score on the fifth,
repeating five times so every development battle is evaluated once. Compare the
mean accuracy, each fold's score, and the majority-class reference. The holdout
is excluded from model selection.


In [ ]:
candidates = [
    Candidate(
        "selected_logistic",
        groups=("selected",),
        model="logistic_regression",
        percentile=100,
        params={"C": 1.0, "max_iter": 2000},
    ),
    Candidate(
        "selected_hist_boosting",
        groups=("selected",),
        model="hist_gradient_boosting",
        percentile=100,
        params={
            "max_iter": 250,
            "learning_rate": 0.06,
            "max_leaf_nodes": 15,
            "min_samples_leaf": 25,
            "l2_regularization": 5.0,
            "early_stopping": False,
        },
    ),
]
results = experiment.compare(candidates)
display(results)
print("Highest mean CV accuracy:", experiment.best_candidate)


## 4. Inspect retained features and fold variation

`cv_accuracy` is the mean of the five validation accuracies. `fold_std` describes
variation between folds; it is not a confidence interval. `dummy_accuracy` is the
majority-class reference on those same folds.

`selected_in_folds` reports how often each input survived preprocessing. A constant
input can be removed even with `percentile=100`. Add any new uniquely named
candidates through `experiment.compare(...)` before opening the holdout.


In [ ]:
for candidate in candidates:
    path = experiment.output_path / f"selection_frequency_{candidate.name}.csv"
    selection = pd.read_csv(path).sort_values(
        ["selected_in_folds", "feature"], ascending=[False, True]
    )
    print(candidate.name)
    display(selection)


## 5. Final holdout evaluation — explicit decision

Leave this **False** while choosing features and models. Set it to **True** only when you have chosen your approach from CV. The chosen pipeline is fitted on the 80% development subset and evaluated on the untouched 20% holdout.

Opening the holdout locks further comparisons in this experiment. Do not tune on its score or repeatedly create new runs to work around the lock; doing so would turn the holdout into another tuning set. The guard is per Python experiment object, not a cross-team access control system.

In [ ]:
EVALUATE_HOLDOUT = False
CHOSEN_CANDIDATE = experiment.best_candidate  # or a specific compared candidate name

if EVALUATE_HOLDOUT:
    holdout = experiment.evaluate_holdout(CHOSEN_CANDIDATE)
    print("Candidate:", holdout["candidate"])
    print("Holdout accuracy:", holdout["accuracy"])
    display(pd.DataFrame(holdout["confusion_matrix"],
                         index=["actual loss", "actual win"], columns=["predicted loss", "predicted win"]))
    display(pd.DataFrame(holdout["classification_report"]).T)
    print("Selected features:", holdout["selected_features"])
else:
    print("Holdout remains unopened. Continue comparing using CV only.")


## 6. Train on all labels and generate a submission (optional)

After evaluating the chosen candidate, set `MAKE_SUBMISSION=True`. The shared
runner refits the locked configuration on all labelled training battles, including
the former holdout, and predicts `test.jsonl`. Its labels are unknown locally.

The sample submission supplies the IDs and row order. Files are saved as
`submission.csv`, `model.joblib`, and `final_fit.json` inside this run's output
folder; nothing is uploaded automatically. The saved pipeline accepts the recorded
numeric input columns, so retain this notebook's feature function for raw battles.


In [ ]:
MAKE_SUBMISSION = False
if MAKE_SUBMISSION:
    submission = experiment.make_submission()
    display(submission.head())
    print("Upload manually:", experiment.output_path / "submission.csv")
else:
    print("Submission generation skipped.")


## 7. Artifacts and experiment notes

The unique ignored run folder contains:

- `experiment.json`: validation settings, data/split hashes, versions, code commit and dirty-tree status.
- `feature_sources.json`: feature function source where available.
- `splits.csv`: saved holdout and development-fold assignments.
- `candidates.json`, `cv_results.csv`, and `selection_frequency_*.csv`: configurations and CV evidence.
- After holdout evaluation: metrics, selected features, and battle-level predictions.
- After final fitting: `model.joblib`, `submission.csv`, and `final_fit.json`.

Record the hypothesis, interpretation, and exact run/commit in your notes. Uncommitted changes are flagged in the manifest; commit the implementation and retain this notebook/configuration before relying on a final result. Saved source snippets help auditing but do not automatically package custom imports or globals.

Only load model files you trust. Saved outputs are local and ignored by Git; commit the notebook/shared code separately and preserve final artifacts with your team.

The current run uses the project validation configuration and all labelled rows.
Keep results from the old standalone grouped/excluded-data workflow separate.


In [ ]:
print("Run folder:", experiment.output_path)
for path in sorted(experiment.output_path.iterdir()):
    print(" -", path.name)
